# Hate Speech Dataset — Shared Preprocessing & Cleaning Notebook

**Goal:** Produce a *single, shared* cleaned text column and a *fixed stratified split* (train/val/test) so every team member trains different models/embeddings on identical inputs.

**Outputs (created by this notebook):**
- `HateSpeechDataset_preprocessed.csv` with columns: `id`, `text`, `label`, `split`
- Summary stats (class balance, text lengths)

**Reproducibility:** This notebook uses a fixed random seed (`SEED = 42`).  


In [ ]:
# Cell 1 — Imports and global settings
import re
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

# Make pandas printouts easier to read
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 120)

SEED = 42  # Fixed seed for reproducible splits


## Cell 1 — What this does
- Imports core libraries (`pandas`, `numpy`, `re`) and the split helper (`train_test_split`) from scikit-learn.
- Sets `SEED = 42` so your train/val/test split is the same for every teammate and every run.
- Adjusts pandas display options for cleaner previews.


In [ ]:
# Cell 2 — Load the raw dataset
# Update this path if your file is in a different location.
IN_PATH = r"/mnt/data/HateSpeechDataset.csv"

df_raw = pd.read_csv(IN_PATH)
print("Raw shape:", df_raw.shape)
df_raw.head()


## Cell 2 — What this does
- Reads the CSV into a DataFrame called `df_raw`.
- Prints its shape (rows × columns) so you can confirm nothing is missing.
- Shows the first 5 rows for a quick sanity check.


In [ ]:
# Cell 3 — Basic schema & missing values check
print("Columns:", df_raw.columns.tolist())
print(df_raw.dtypes)

# Missing values per column
df_raw.isna().sum()


## Cell 3 — What this does
- Prints column names and data types (e.g., whether labels are strings or numeric).
- Counts missing values in each column. Missing text or label values must be handled before training.


In [ ]:
# Cell 4 — Remove accidental duplicated header rows (data issue)
# Some copies of this dataset contain duplicated header rows inside the file.
# Those rows look like: Content='Content', Label='Label', Content_int='Content_int'
# We'll identify them by checking if the label literally equals the string 'Label'.

df = df_raw.copy()

mask_bad_header = df["Label"].astype(str).str.strip().str.lower().eq("label")
num_bad = int(mask_bad_header.sum())
print("Duplicated header-like rows detected:", num_bad)

df = df.loc[~mask_bad_header].reset_index(drop=True)
print("After removing header-like rows:", df.shape)


## Cell 4 — What this does
- Fixes a common dataset formatting problem: **duplicated header rows embedded as data**.
- Detects those rows by checking if `Label` equals the string `"Label"` (case-insensitive).
- Removes them and resets the index to keep row numbering clean.


In [ ]:
# Cell 5 — Normalize and validate labels
# Goal: create a clean integer label column `label` with values in {0,1}.

# Many datasets store labels as strings; we coerce safely.
df["Label"] = df["Label"].astype(str).str.strip()

# Convert to numeric; invalid values become NaN
df["label"] = pd.to_numeric(df["Label"], errors="coerce")

# Keep only binary labels 0/1
before = len(df)
df = df[df["label"].isin([0, 1])].copy()
after = len(df)

df["label"] = df["label"].astype(int)

print(f"Rows before label filtering: {before:,}")
print(f"Rows after  label filtering: {after:,}")
print("Label counts:")
print(df["label"].value_counts())
print("Positive rate:", df["label"].mean())


## Cell 5 — What this does
- Strips whitespace in the `Label` column.
- Converts `Label` to a numeric `label` column using `pd.to_numeric(..., errors="coerce")`.
  - Any non-numeric garbage becomes `NaN` and is filtered out.
- Keeps only rows where `label` is **0 or 1** (binary classification).
- Prints class counts and the positive rate (helps you decide metrics like F1).


In [ ]:
# Cell 6 — Define the shared text cleaning function
# This function is the shared preprocessing baseline for all models and embeddings.

URL_RE = re.compile(r"""\b(?:https?://|www\.)\S+\b""", re.IGNORECASE)
EMAIL_RE = re.compile(r"""\b[\w\.-]+@[\w\.-]+\.\w+\b""", re.IGNORECASE)

# NOTE: We avoid variable-width lookbehind (not supported by Python re).
# We mask @mentions after masking emails, so @ inside emails won't be affected.
USER_RE = re.compile(r"""@[A-Za-z0-9_]+\b""")

def clean_text(s: str) -> str:
    """Clean a single text string into a normalized form.

    Steps (shared baseline):
    1) Lowercase
    2) Normalize curly quotes
    3) Mask URLs/emails/@mentions with <URL>/<EMAIL>/<USER>
    4) Remove HTML entities like &amp;
    5) Replace non-alphanumeric (except apostrophes) with spaces
    6) Collapse extra spaces
    """
    if pd.isna(s):
        return ""
    s = str(s)

    # Lowercase
    s = s.lower()

    # Normalize common unicode quotes
    s = s.replace("’", "'").replace("‘", "'").replace("“", '"').replace("”", '"')

    # Mask entities
    s = URL_RE.sub(" <URL> ", s)
    s = EMAIL_RE.sub(" <EMAIL> ", s)
    s = USER_RE.sub(" <USER> ", s)

    # Remove common HTML entities (extend if needed)
    s = s.replace("&amp;", " and ").replace("&lt;", "<").replace("&gt;", ">")

    # Keep letters, numbers, apostrophes; turn the rest into spaces
    s = re.sub(r"[^a-z0-9']+", " ", s)

    # Collapse multiple spaces
    s = re.sub(r"\s+", " ", s).strip()

    return s


## Cell 6 — What this does
- Creates compiled regular expressions for URLs, emails, and @mentions.
- Defines `clean_text()` which transforms raw text into a stable, comparable format.
- Uses masking tokens (`<URL>`, `<EMAIL>`, `<USER>`) instead of deleting information.  
  This keeps potentially predictive cues while reducing vocabulary noise.
- Keeps apostrophes so contractions like *don't* remain meaningful.


In [ ]:
# Cell 7 — Apply cleaning to the Content column
# We build a standardized `text` column from df['Content'].

# Ensure the expected column exists
assert "Content" in df.columns, "Expected a 'Content' column in the dataset."

df["text"] = df["Content"].apply(clean_text)

# Drop rows that became empty after cleaning (optional but usually helpful)
before = len(df)
df = df[df["text"].str.len() > 0].copy()
after = len(df)

print(f"Rows before dropping empty text: {before:,}")
print(f"Rows after  dropping empty text: {after:,}")

df[["Content", "text", "label"]].head()


## Cell 7 — What this does
- Applies the shared `clean_text()` function to `df['Content']` and stores the result in `df['text']`.
- Optionally drops rows that become empty (e.g., if the original row had only symbols/links).
- Displays a side-by-side preview of original vs cleaned text.


In [ ]:
# Cell 8 — Quick dataset exploration (EDA): text length & class balance
# These stats help justify preprocessing choices in the report.

df["n_chars"] = df["text"].str.len()
df["n_tokens"] = df["text"].str.split().apply(len)

print("Class balance:")
print(df["label"].value_counts())
print("\nToken length (n_tokens) quantiles:")
print(df["n_tokens"].quantile([0.5, 0.9, 0.95, 0.99]).to_string())


## Cell 8 — What this does
- Computes two simple length features:
  - `n_chars`: number of characters in the cleaned text
  - `n_tokens`: number of whitespace-separated tokens
- Prints:
  - class distribution (imbalance awareness)
  - token-length quantiles (useful to pick `max_len` for RNN/LSTM/GRU, or to justify truncation)


In [ ]:
# Cell 9 — Create a shared stratified train/val/test split
# Stratification keeps label proportions consistent across splits.

# Step 1: split off test (10%)
train_val, test = train_test_split(
    df,
    test_size=0.10,
    random_state=SEED,
    stratify=df["label"]
)

# Step 2: split train vs val from remaining 90% -> val is 10% overall
# val_fraction_of_train_val = 0.10 / 0.90
val_size = 0.10 / 0.90

train, val = train_test_split(
    train_val,
    test_size=val_size,
    random_state=SEED,
    stratify=train_val["label"]
)

train = train.copy()
val = val.copy()
test = test.copy()

train["split"] = "train"
val["split"] = "val"
test["split"] = "test"

df_split = pd.concat([train, val, test], axis=0).reset_index(drop=True)

print("Split sizes:")
print(df_split["split"].value_counts())
print("\nLabel rate by split:")
print(df_split.groupby("split")["label"].mean())


## Cell 9 — What this does
- Produces a **fixed 80/10/10 stratified split**:
  - First split: 90% (train+val) / 10% (test)
  - Second split: train vs val inside the 90% so that val becomes 10% overall
- Stratification ensures each split keeps similar label proportions.
- Adds a `split` column so every teammate can filter consistently:
  - `split == "train"` for training
  - `split == "val"` for hyperparameter tuning / early stopping
  - `split == "test"` for final evaluation


In [ ]:
# Cell 10 — Export the shared cleaned dataset
OUT_PATH = r"/mnt/data/HateSpeechDataset_preprocessed.csv"

# Keep only the columns needed for downstream modeling + reproducibility
export_cols = ["text", "label", "split"]

df_out = df_split[export_cols].copy()
df_out.insert(0, "id", np.arange(len(df_out), dtype=int))

df_out.to_csv(OUT_PATH, index=False)
print("Saved:", OUT_PATH)
print("Export shape:", df_out.shape)
df_out.head()


## Cell 10 — What this does
- Builds a compact export file with just the essentials:
  - `id`: stable row identifier (0..N-1)
  - `text`: cleaned text
  - `label`: 0/1 integer
  - `split`: train/val/test
- Saves to CSV so every teammate can load the same artifact.


# Next steps for each team member (how to use the output)

- **Traditional ML (Logistic Regression / SVM / Random Forest)**  
  Use `text` as input to TF-IDF vectorizer, train on `split == "train"`, tune on `split == "val"`, report on `split == "test"`.

- **Sequence models (RNN / LSTM / GRU)**  
  Tokenize `text` consistently (e.g., `text.split()` or a shared tokenizer).  
  Pick `max_len` based on quantiles from Cell 8 (e.g., 95th percentile).

- **Embedding variants (Word2Vec Skip-gram/CBOW, GloVe, FastText)**  
  Keep the **same cleaned text**; only change the embedding method and model architecture.

**Important:** Do not change this preprocessing once experiments start, or comparisons become invalid.
